# Experiment: Local Alpha Sandbox

Objective:
- Train one small local tabular model on the current trading features.
- Understand which features most strongly explain the current ranker's top bucket.
- Keep the workflow simple enough to run and modify on a single desktop GPU or CPU.


## What This Sandbox Is

This first notebook is deliberately modest. It does **not** try to predict future returns yet.
Instead, it trains a small classifier to mimic the current ranker's `top 20%` names using the latest feature set.

Why this is useful:
- it proves the local training loop works
- it shows feature importance and model behavior
- it gives us a clean place to swap in better labels later, like forward alpha or event outcomes


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

SEED = 7
np.random.seed(SEED)
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
FEATURES_CSV = ROOT / 'trading' / 'data' / 'features' / 'latest_features.csv'
RANKS_CSV = ROOT / 'trading' / 'data' / 'ranks' / 'latest_ranks.csv'
MODEL_CONFIG = ROOT / 'trading' / 'config' / 'alpha_model.json'

FEATURES_CSV, RANKS_CSV, MODEL_CONFIG


## Plan

- Hypothesis: a small tree-based model can learn the current ranker's top bucket from the engineered features.
- First label: `top_bucket_target = 1` for names in the top 20% of `final_alpha_score`.
- Metrics: ROC AUC, class report, feature importance, and a quick review of false positives / false negatives.
- Follow-up: replace the target with forward alpha or event outcomes once more labels are available.


In [ ]:
features = pd.read_csv(FEATURES_CSV)
ranks = pd.read_csv(RANKS_CSV)
weights = json.loads(MODEL_CONFIG.read_text())['weights']

df = features.merge(
    ranks[['ticker', 'rank', 'final_alpha_score', 'eligible']],
    on='ticker',
    how='left',
    suffixes=('', '_rank')
)

df['nowcast_source_mix'] = df['nowcast_source_mix'].fillna('none')
df['eligible'] = df['eligible'].fillna(df.get('eligible_rank')).fillna(False).astype(str).str.lower().eq('true')
df['top_bucket_target'] = (df['final_alpha_score'] >= df['final_alpha_score'].quantile(0.80)).astype(int)

df[['ticker', 'rank', 'final_alpha_score', 'top_bucket_target']].head()


In [ ]:
feature_columns = [
    'quality_score',
    'growth_score',
    'value_score',
    'alt_momentum_score',
    'peer_relative_score',
    'proxy_inferred_score',
    'confidence',
    'fundamental_confidence',
    'value_confidence',
    'alt_confidence',
    'peer_confidence',
    'proxy_confidence',
    'structural_anomaly_penalty',
    'valuation_context_strength',
    'valuation_method_strength',
    'revenue_cagr_3y',
    'fcf_cagr_3y',
    'roic',
    'fcf_margin',
    'asset_turnover',
    'debt_to_ebitda',
    'risk_penalty',
    'liquidity_penalty',
    'days_to_earnings',
    'nowcast_direct_source_count',
    'nowcast_proxy_source_count',
    'nowcast_proxy_share',
]

categorical_columns = ['nowcast_source_mix']
target_column = 'top_bucket_target'

train_df = df[feature_columns + categorical_columns + [target_column, 'ticker', 'rank', 'final_alpha_score']].copy()
train_df.shape


In [ ]:
X = train_df[feature_columns + categorical_columns]
y = train_df[target_column]

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
])

preprocess = ColumnTransformer([
    ('num', numeric_transformer, feature_columns),
    ('cat', categorical_transformer, categorical_columns),
], remainder='drop', verbose_feature_names_out=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=4,
    max_iter=250,
    min_samples_leaf=8,
    random_state=SEED,
)

pipeline = Pipeline([
    ('preprocess', preprocess),
    ('model', model),
])

pipeline.fit(X_train, y_train)
pred_proba = pipeline.predict_proba(X_test)[:, 1]
pred_label = (pred_proba >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, pred_proba)
roc_auc


In [ ]:
print(classification_report(y_test, pred_label, digits=3))


In [ ]:
perm = permutation_importance(
    pipeline,
    X_test,
    y_test,
    n_repeats=10,
    random_state=SEED,
    scoring='roc_auc',
)

importance_df = pd.DataFrame({
    'feature': feature_columns + categorical_columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)

importance_df.head(15)


In [ ]:
scored = train_df[['ticker', 'rank', 'final_alpha_score', 'top_bucket_target']].copy()
scored['model_top_bucket_proba'] = pipeline.predict_proba(X)[:, 1]
scored = scored.sort_values('model_top_bucket_proba', ascending=False)

scored.head(15)


## Results

- If ROC AUC is meaningfully above `0.5`, the sandbox is learning something coherent from the current feature engineering.
- The feature-importance table tells you which parts of the pipeline are actually driving the learned top-bucket behavior.
- Do not treat this as tradable alpha yet. It is a teaching scaffold and model-debugging sandbox.


In [ ]:
result = {
    'seed': SEED,
    'row_count': int(len(train_df)),
    'positive_rate': float(train_df['top_bucket_target'].mean()),
    'roc_auc': float(roc_auc),
    'top_features': importance_df.head(10)['feature'].tolist(),
    'top_predicted_names': scored.head(10)['ticker'].tolist(),
    'alpha_weights': weights,
}
result


## Next Steps

- Swap the label from `top_bucket_target` to a forward-looking target, such as realized 20-day alpha or event-study outcomes.
- Compare `HistGradientBoostingClassifier` with `XGBoost` or `CatBoost` if those are installed locally.
- Run the same notebook on sleeve-specific cohorts like `Nowcast Ablation` or `Earnings Drift Lab`.
- Add SHAP or partial-dependence plots once the basic loop feels comfortable.
